# Serizay et al. (2020) Tissue-Resolved ATAC-seq Reference Download

Downloads and verifies canonical tissue-resolved young adult *C. elegans* ATAC-seq coverage BigWigs (Intestine, Neurons, Germline) from NCBI GEO (`GSE141210` / `GSE141213`) into `notebooks/reference/`.

---

### Required Inputs (Remote NCBI GEO Provenance)

| Track Name | GEO Accession | Remote Source URL | Size (Bytes) | SHA-256 (16-char) |
|---|---|---|---|---|
| **Intestine ATAC** | GSM4197242 | `https://ftp.ncbi.nlm.nih.gov/.../GSM4197242_Intestine_YA_ATAC_pe_rep1.bw` | 84,721,022 | `c8d3e5779d4b3a6d` |
| **Neurons ATAC** | GSM4197239 | `https://ftp.ncbi.nlm.nih.gov/.../GSM4197239_Neurons_YA_ATAC_pe_rep1.bw` | 33,807,035 | `9a8fdef03a1fe262` |
| **Germline ATAC** | GSM4197238 | `https://ftp.ncbi.nlm.nih.gov/.../GSM4197238_Germline_YA_ATAC_pe_rep1.bw` | 46,662,225 | `ec35e65f3e0514fb` |

### Expected Outputs

| Output Role | Target File | Description |
|---|---|---|
| **Intestine ATAC Track** | `notebooks/reference/GSM4197242_Intestine_YA_ATAC_pe_rep1.bw` | FACS-sorted young adult intestine ATAC-seq coverage (WBcel235/ce11) |
| **Neurons ATAC Track** | `notebooks/reference/GSM4197239_Neurons_YA_ATAC_pe_rep1.bw` | FACS-sorted young adult neuron ATAC-seq coverage (WBcel235/ce11) |
| **Germline ATAC Track** | `notebooks/reference/GSM4197238_Germline_YA_ATAC_pe_rep1.bw` | FACS-sorted young adult germline ATAC-seq coverage (WBcel235/ce11) |

---

### Data Provenance & Biological Context
- **Publication**: Serizay, J. et al. (2020). *Distinct chromatin accessibility profiles define tissue-specific gene expression in Caenorhabditis elegans*. **Genome Research**, 30(10):1452–1464. [doi:10.1101/gr.266205.120](https://doi.org/10.1101/gr.266205.120).
- **Series**: NCBI GEO [GSE141210](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE141210) (SubSeries [GSE141213](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE141213)).
- **Method**: Tissue-specific nuclei from young adult hermaphrodites isolated via FACS sorting (GFP-marked promoters: *elt-2p::GFP* for intestine, *rgef-1p::GFP* for pan-neuronal, *pie-1p::GFP* for germline), subjected to Omni-ATAC, paired-end sequenced, and mapped to the *C. elegans* reference assembly (WBcel235 / ce11).
- **Normalization**: Each track's canonical divisor is extracted directly from its BigWig header as `sumData / nBasesCovered` (~1.4962 across all three tracks).


### 1. Configuration & Path Resolution

Resolves repository root, verifies `notebooks/reference/` target directory, and defines NCBI GEO download manifests.

In [1]:
import hashlib
import os
import sys
import urllib.request
from pathlib import Path
import pandas as pd
import pyBigWig

NOTEBOOKS_DIR = Path.cwd()
REPO_ROOT = NOTEBOOKS_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reference directory
REF_DIR = NOTEBOOKS_DIR / "reference"
REF_DIR.mkdir(parents=True, exist_ok=True)

# NCBI GEO Download Configurations
ATAC_DOWNLOAD_CONFIGS = {
    "Intestine_ATAC": {
        "filename": "GSM4197242_Intestine_YA_ATAC_pe_rep1.bw",
        "url": "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4197nnn/GSM4197242/suppl/GSM4197242_Intestine_YA_ATAC_pe_rep1.bw",
        "size_bytes": 84721022,
        "sha256_16": "c8d3e5779d4b3a6d",
    },
    "Neurons_ATAC": {
        "filename": "GSM4197239_Neurons_YA_ATAC_pe_rep1.bw",
        "url": "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4197nnn/GSM4197239/suppl/GSM4197239_Neurons_YA_ATAC_pe_rep1.bw",
        "size_bytes": 33807035,
        "sha256_16": "9a8fdef03a1fe262",
    },
    "Germline_ATAC": {
        "filename": "GSM4197238_Germline_YA_ATAC_pe_rep1.bw",
        "url": "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4197nnn/GSM4197238/suppl/GSM4197238_Germline_YA_ATAC_pe_rep1.bw",
        "size_bytes": 46662225,
        "sha256_16": "ec35e65f3e0514fb",
    },
}

print(f"Repository Root:     {REPO_ROOT}")
print(f"Reference Directory: {REF_DIR}")
for trk, cfg in ATAC_DOWNLOAD_CONFIGS.items():
    p = REF_DIR / cfg["filename"]
    status = "✓ Found" if p.is_file() else "✗ Missing"
    print(f"  {trk:15s}: {cfg['filename']} [{status}]")


Repository Root:     /Users/dan/Code/UMassMed_Projects/Cut_And_Run_Figures
Reference Directory: /Users/dan/Code/UMassMed_Projects/Cut_And_Run_Figures/notebooks/reference
  Intestine_ATAC : GSM4197242_Intestine_YA_ATAC_pe_rep1.bw [✗ Missing]
  Neurons_ATAC   : GSM4197239_Neurons_YA_ATAC_pe_rep1.bw [✗ Missing]
  Germline_ATAC  : GSM4197238_Germline_YA_ATAC_pe_rep1.bw [✗ Missing]


### 2. Download from NCBI GEO & Verify SHA-256 Integrity

Fetches each BigWig stream via HTTP if missing or incomplete, and validates file size and SHA-256 hash prefix.

In [2]:
print(f"Reference Directory: {REF_DIR}")
for trk, cfg in ATAC_DOWNLOAD_CONFIGS.items():
    dest = REF_DIR / cfg["filename"]
    if dest.is_file() and dest.stat().st_size == cfg["size_bytes"]:
        h = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
        status = "Verified" if h == cfg["sha256_16"] else "Checksum Mismatch"
        print(f"  ✓ {trk:15s}: {dest.name} ({dest.stat().st_size:,} bytes, sha256_16: {h}) [{status}]")
    else:
        print(f"  ⬇ Downloading {trk} from GEO ({cfg['size_bytes'] / 1e6:.1f} MB)...")
        req = urllib.request.Request(cfg["url"], headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(dest, "wb") as f_out:
            while chunk := resp.read(1024 * 1024):
                f_out.write(chunk)
        h = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
        print(f"  ✓ Downloaded {dest.name} ({dest.stat().st_size:,} bytes, sha256_16: {h})")


Reference Directory: /Users/dan/Code/UMassMed_Projects/Cut_And_Run_Figures/notebooks/reference
  ⬇ Downloading Intestine_ATAC from GEO (84.7 MB)...
  ✓ Downloaded GSM4197242_Intestine_YA_ATAC_pe_rep1.bw (84,721,022 bytes, sha256_16: c8d3e5779d4b3a6d)
  ⬇ Downloading Neurons_ATAC from GEO (33.8 MB)...
  ✓ Downloaded GSM4197239_Neurons_YA_ATAC_pe_rep1.bw (33,807,035 bytes, sha256_16: 9a8fdef03a1fe262)
  ⬇ Downloading Germline_ATAC from GEO (46.7 MB)...
  ✓ Downloaded GSM4197238_Germline_YA_ATAC_pe_rep1.bw (46,662,225 bytes, sha256_16: ec35e65f3e0514fb)


### 3. BigWig Track Validation & Canonical Divisors

Inspects BigWig headers using `pyBigWig` to confirm genome coordinate definitions (WBcel235 / ce11 chromosomes), coverage base counts, and compute the canonical normalization divisor (`sumData / nBasesCovered`).

In [3]:
track_summary = []
for trk, cfg in ATAC_DOWNLOAD_CONFIGS.items():
    dest = REF_DIR / cfg["filename"]
    bw = pyBigWig.open(str(dest))
    hdr = bw.header()
    n_chroms = len(bw.chroms())
    n_bases = hdr["nBasesCovered"]
    sum_data = hdr["sumData"]
    divisor = sum_data / n_bases if n_bases > 0 else 0.0
    bw.close()
    track_summary.append({
        "track": trk,
        "filename": dest.name,
        "size_mb": round(dest.stat().st_size / (1024 * 1024), 2),
        "n_chroms": n_chroms,
        "nBasesCovered": n_bases,
        "sumData": sum_data,
        "genome_mean_divisor": divisor,
    })

df_summary = pd.DataFrame(track_summary)
display(df_summary)


,track,filename,size_mb,n_chroms,nBasesCovered,sumData,genome_mean_divisor
0,Intestine_ATAC,GSM4197242_Intestine_YA_ATAC_pe_rep1.bw,80.80,6,100252146,150000007,1.496227
1,Neurons_ATAC,GSM4197239_Neurons_YA_ATAC_pe_rep1.bw,32.24,6,100251705,149999990,1.496234
2,Germline_ATAC,GSM4197238_Germline_YA_ATAC_pe_rep1.bw,44.50,6,100252039,149999996,1.496229
